In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt  # kept for compatibility; all final charts below use Plotly
from sklearn.decomposition import PCA
from pathlib import Path
import zipfile

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# ------------------------------------------------------------
# Global presentation settings
# ------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PLOT_TEMPLATE = "plotly_white"
COLOR_SEQUENCE = px.colors.qualitative.Bold

TEAM_NUMBER = "7"

# Optional: set this manually if your CSVs are in a different folder.
# Example on Windows: DATA_DIR_OVERRIDE = r"C:\\Users\\dell\\Downloads\\QMP Case Study 2026"
DATA_DIR_OVERRIDE = None


def _find_data_dir():
    """Find the folder containing ticks_mon.csv in a local or notebook-grading environment."""
    candidates = []

    if DATA_DIR_OVERRIDE is not None:
        candidates.append(Path(DATA_DIR_OVERRIDE).expanduser())

    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd / "QMP Case Study 2026",
        cwd / "data",
        cwd / "qmp_case_calc" / "QMP Case Study 2026",
        Path("/mnt/data/qmp_case_calc/QMP Case Study 2026"),
        Path("/mnt/data"),
    ])

    for d in candidates:
        if (d / "ticks_mon.csv").exists():
            return d

    # If the zip is next to the notebook, auto-extract it once.
    zip_candidates = [cwd / "QMP Case Study 2026.zip", Path("/mnt/data/QMP Case Study 2026.zip")]
    for z in zip_candidates:
        if z.exists():
            extract_dir = z.parent / "qmp_case_calc"
            extract_dir.mkdir(exist_ok=True)
            with zipfile.ZipFile(z, "r") as zip_ref:
                zip_ref.extractall(extract_dir)
            for found in extract_dir.rglob("ticks_mon.csv"):
                return found.parent

    # Last fallback: limited recursive search from current directory.
    for root in [cwd, Path("/mnt/data")]:
        if root.exists():
            for found in root.rglob("ticks_mon.csv"):
                return found.parent

    raise FileNotFoundError(
        "Could not find ticks_mon.csv. Put the notebook in the same folder as the CSV files, "
        "keep QMP Case Study 2026.zip next to it, or set DATA_DIR_OVERRIDE manually."
    )


DATA_DIR = _find_data_dir()
print("Using data directory:", DATA_DIR.resolve())


def load_tick_file(filename: str) -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / filename)


def style_figure(fig, title=None, height=520):
    """Apply one consistent presentation-ready style to every Plotly figure."""
    fig.update_layout(
        template=PLOT_TEMPLATE,
        height=height,
        title=dict(text=title if title else fig.layout.title.text, x=0.02, xanchor="left"),
        font=dict(size=13),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=60, r=40, t=90, b=60),
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)")
    return fig

Using data directory: C:\Users\dell\Downloads\qmp_case_calc\QMP Case Study 2026


In [2]:
# ========================================
# (a) Load Monday ticks
# ========================================

ticks_mon = load_tick_file("ticks_mon.csv")

returns_mon = (
    ticks_mon
    .pivot(
        index="timestamp",
        columns="asset_id",
        values="mid_return"
    )
    .dropna()
)

print("Shape:", returns_mon.shape)
print("Assets:", list(returns_mon.columns))

Shape: (28245, 12)
Assets: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12']


In [3]:
# ========================================
# Fit PCA
# ========================================

pca_mon = PCA().fit(
    returns_mon.values
)

eigenvalues = pca_mon.explained_variance_

explained_var_ratio = (
    pca_mon.explained_variance_ratio_
)

In [4]:
eig_table = pd.DataFrame({
    "PC": range(1, len(eigenvalues)+1),
    "Eigenvalue": eigenvalues,
    "Explained Variance Ratio": explained_var_ratio,
    "Cumulative Variance":
        np.cumsum(explained_var_ratio)
})

display(
    eig_table.style.format({
        "Eigenvalue": "{:.6e}",
        "Explained Variance Ratio": "{:.2%}",
        "Cumulative Variance": "{:.2%}"
    })
)

print(
    f"Variance explained by first 3 PCs = "
    f"{100*np.sum(explained_var_ratio[:3]):.2f}%"
)

,PC,Eigenvalue,Explained Variance Ratio,Cumulative Variance
0,1,6.093245e-06,55.52%,55.52%
1,2,3.004494e-06,27.38%,82.90%
2,3,1.066268e-06,9.72%,92.62%
3,4,9.275436e-08,0.85%,93.46%
4,5,9.224297e-08,0.84%,94.30%
5,6,9.149816e-08,0.83%,95.14%
6,7,9.018086e-08,0.82%,95.96%
7,8,8.983593e-08,0.82%,96.78%
8,9,8.915401e-08,0.81%,97.59%
9,10,8.863223e-08,0.81%,98.40%


Variance explained by first 3 PCs = 92.62%


In [5]:
# Presentation scree plot: eigenvalue bars + cumulative explained variance line.
scree_df = eig_table.copy()
scree_df["PC_label"] = scree_df["PC"].apply(lambda x: f"PC{x}")
scree_df["Explained Variance %"] = 100 * scree_df["Explained Variance Ratio"]
scree_df["Cumulative Variance %"] = 100 * scree_df["Cumulative Variance"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=scree_df["PC_label"],
        y=scree_df["Eigenvalue"],
        name="Eigenvalue",
        marker=dict(color=scree_df["Eigenvalue"], colorscale="Viridis", showscale=False),
        hovertemplate="%{x}<br>Eigenvalue=%{y:.3e}<extra></extra>",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=scree_df["PC_label"],
        y=scree_df["Cumulative Variance %"],
        name="Cumulative variance",
        mode="lines+markers+text",
        text=[f"{v:.1f}%" if pc <= 3 else "" for pc, v in zip(scree_df["PC"], scree_df["Cumulative Variance %"])],
        textposition="top center",
        line=dict(width=4),
        marker=dict(size=9),
        hovertemplate="%{x}<br>Cumulative variance=%{y:.2f}%<extra></extra>",
    ),
    secondary_y=True,
)

fig.add_vrect(
    x0=-0.5,
    x1=2.5,
    fillcolor="rgba(0, 180, 120, 0.12)",
    line_width=0,
    annotation_text="Chosen factor block: K = 3",
    annotation_position="top left",
)

fig.update_yaxes(title_text="Eigenvalue", secondary_y=False)
fig.update_yaxes(title_text="Cumulative explained variance (%)", secondary_y=True, range=[0, 105])
fig.update_xaxes(title_text="Principal component")
style_figure(fig, "Part 1(a): Scree plot with cumulative explained variance", height=560).show()

In [6]:
def select_n_components(
    returns_matrix,
    eigenvalues,
    n_resamples=500,
    seed=0
):

    np.random.seed(seed)

    p = len(eigenvalues)

    mean_eig = np.mean(
        eigenvalues
    )

    # Kaiser

    kaiser = np.sum(
        eigenvalues > mean_eig
    )

    # Broken Stick

    broken_stick_threshold = []

    for j in range(1, p+1):

        threshold = mean_eig * sum(
            1/i for i in range(j, p+1)
        )

        broken_stick_threshold.append(
            threshold
        )

    broken_stick_threshold = np.array(
        broken_stick_threshold
    )

    broken_stick = np.sum(
        eigenvalues > broken_stick_threshold
    )

    # Parallel Analysis

    centered = (
        returns_matrix
        - returns_matrix.mean(axis=0)
    )

    resampled_eigs = np.zeros(
        (n_resamples, p)
    )

    for r in range(n_resamples):

        shuffled = centered.copy()

        for c in range(
            shuffled.shape[1]
        ):

            shuffled[:, c] = np.random.permutation(
                shuffled[:, c]
            )

        eig_rand = PCA().fit(
            shuffled
        ).explained_variance_

        resampled_eigs[r] = eig_rand

    parallel_threshold = np.percentile(
        resampled_eigs,
        95,
        axis=0
    )

    parallel = np.sum(
        eigenvalues > parallel_threshold
    )

    verdicts = {
        "kaiser": int(kaiser),
        "broken_stick": int(broken_stick),
        "parallel": int(parallel)
    }

    return (
        verdicts,
        broken_stick_threshold,
        parallel_threshold
    )

In [7]:
verdicts, broken_stick_threshold, parallel_threshold = (
    select_n_components(
        returns_mon.values,
        eigenvalues
    )
)

print(
    "Component-count verdicts:",
    verdicts
)

Component-count verdicts: {'kaiser': 3, 'broken_stick': 3, 'parallel': 3}


In [8]:
# Broken-stick test: observed eigenvalues versus random-share benchmark.
bs_df = pd.DataFrame({
    "PC": [f"PC{i}" for i in range(1, 13)],
    "Observed eigenvalue": eigenvalues,
    "Broken-stick threshold": broken_stick_threshold,
})
bs_df["Retained"] = bs_df["Observed eigenvalue"] > bs_df["Broken-stick threshold"]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bs_df["PC"],
    y=bs_df["Observed eigenvalue"],
    mode="lines+markers",
    name="Observed eigenvalue",
    line=dict(width=4),
    marker=dict(size=9),
    hovertemplate="%{x}<br>Observed=%{y:.3e}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=bs_df["PC"],
    y=bs_df["Broken-stick threshold"],
    mode="lines+markers",
    name="Broken-stick threshold",
    line=dict(width=3, dash="dash"),
    marker=dict(size=8, symbol="diamond"),
    hovertemplate="%{x}<br>Threshold=%{y:.3e}<extra></extra>",
))

for idx, retained in enumerate(bs_df["Retained"]):
    if retained:
        fig.add_vrect(x0=idx-0.45, x1=idx+0.45, fillcolor="rgba(60, 180, 75, 0.10)", line_width=0)

fig.update_xaxes(title_text="Principal component")
fig.update_yaxes(title_text="Eigenvalue")
style_figure(fig, "Part 1(a): Broken-stick test keeps the components above the benchmark", height=520).show()

In [9]:
# Parallel analysis: compare observed eigenvalues with shuffled-column 95th percentile.
pa_df = pd.DataFrame({
    "PC": [f"PC{i}" for i in range(1, 13)],
    "Observed eigenvalue": eigenvalues,
    "Parallel-analysis 95th percentile": parallel_threshold,
})
pa_df["Retained"] = pa_df["Observed eigenvalue"] > pa_df["Parallel-analysis 95th percentile"]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pa_df["PC"],
    y=pa_df["Observed eigenvalue"],
    mode="lines+markers",
    name="Observed eigenvalue",
    line=dict(width=4),
    marker=dict(size=9),
    hovertemplate="%{x}<br>Observed=%{y:.3e}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=pa_df["PC"],
    y=pa_df["Parallel-analysis 95th percentile"],
    mode="lines+markers",
    name="95th percentile under shuffled null",
    line=dict(width=3, dash="dot"),
    marker=dict(size=8, symbol="x"),
    hovertemplate="%{x}<br>Null 95th percentile=%{y:.3e}<extra></extra>",
))

for idx, retained in enumerate(pa_df["Retained"]):
    if retained:
        fig.add_vrect(x0=idx-0.45, x1=idx+0.45, fillcolor="rgba(255, 193, 7, 0.13)", line_width=0)

fig.update_xaxes(title_text="Principal component")
fig.update_yaxes(title_text="Eigenvalue")
style_figure(fig, "Part 1(a): Parallel analysis confirms the meaningful factor count", height=520).show()

In [10]:
criteria_table = pd.DataFrame({
    "Criterion": [
        "Kaiser",
        "Broken Stick",
        "Parallel Analysis"
    ],
    "Suggested K": [
        verdicts["kaiser"],
        verdicts["broken_stick"],
        verdicts["parallel"]
    ]
})

display(criteria_table)

fig = px.bar(
    criteria_table,
    x="Criterion",
    y="Suggested K",
    text="Suggested K",
    color="Criterion",
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Part 1(a): Component-count verdicts agree on K = 3",
)
fig.update_traces(textposition="outside")
fig.update_yaxes(title_text="Suggested number of components", dtick=1, range=[0, max(criteria_table["Suggested K"]) + 1])
fig.update_xaxes(title_text="Selection criterion")
style_figure(fig, height=460).show()

,Criterion,Suggested K
0,Kaiser,3
1,Broken Stick,3
2,Parallel Analysis,3


### Component Selection Results

The scree plot exhibits a sharp decline after the first three principal components, indicating that most of the variability in the return matrix is captured by a small number of factors.

The top three components explain approximately:

- PC1: 55.52% of total variance
- PC2: 27.38% of total variance
- PC3: 9.72% of total variance

Together, the first three principal components explain approximately 92.62% of the total variance.

The component-selection criteria produce the following verdicts:

- Kaiser criterion: K = 3
- Broken-stick criterion: K = 3
- Parallel analysis (500 shuffled resamples, 95th percentile threshold): K = 3

Since all criteria agree, we retain K = 3 principal components for all downstream analysis.

The scree plot also displays a clear elbow after the third component, providing additional visual support for this choice.

Part 1 b

In [11]:
########################################
# (b) Plot loadings + rotation ambiguity
########################################

loadings = pca_mon.components_[:2].T

print(loadings.shape)

(12, 2)


PC1 vs PC2 Loading Plot

In [12]:
# Interactive loading map. The dashed spokes make the 12-gon geometry easier to explain.
loadings_df = pd.DataFrame({
    "asset_id": returns_mon.columns,
    "PC1 loading": loadings[:, 0],
    "PC2 loading": loadings[:, 1],
})
loadings_df["radius"] = np.sqrt(loadings_df["PC1 loading"]**2 + loadings_df["PC2 loading"]**2)
loadings_df["angle_deg"] = np.degrees(np.arctan2(loadings_df["PC2 loading"], loadings_df["PC1 loading"]))

fig = go.Figure()

for _, row in loadings_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[0, row["PC1 loading"]],
        y=[0, row["PC2 loading"]],
        mode="lines",
        line=dict(width=1, dash="dot"),
        opacity=0.45,
        showlegend=False,
        hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(
    x=loadings_df["PC1 loading"],
    y=loadings_df["PC2 loading"],
    mode="markers+text",
    text=loadings_df["asset_id"],
    textposition="top center",
    marker=dict(
        size=14,
        color=loadings_df["angle_deg"],
        colorscale="Turbo",
        showscale=True,
        colorbar=dict(title="Angle°"),
        line=dict(width=1, color="white"),
    ),
    customdata=np.stack([loadings_df["radius"], loadings_df["angle_deg"]], axis=-1),
    hovertemplate="%{text}<br>PC1=%{x:.4f}<br>PC2=%{y:.4f}<br>Radius=%{customdata[0]:.4f}<br>Angle=%{customdata[1]:.1f}°<extra></extra>",
    name="Asset loading",
))

# Connect assets in label order to visually emphasize the approximate polygon/lattice structure.
ordered = loadings_df.sort_values("asset_id")
fig.add_trace(go.Scatter(
    x=list(ordered["PC1 loading"]) + [ordered["PC1 loading"].iloc[0]],
    y=list(ordered["PC2 loading"]) + [ordered["PC2 loading"].iloc[0]],
    mode="lines",
    line=dict(width=2),
    opacity=0.35,
    name="Asset-order polygon",
    hoverinfo="skip",
))

fig.add_hline(y=0, line_width=1, line_dash="dash", opacity=0.6)
fig.add_vline(x=0, line_width=1, line_dash="dash", opacity=0.6)
fig.update_xaxes(title_text="PC1 loading", zeroline=True)
fig.update_yaxes(title_text="PC2 loading", scaleanchor="x", scaleratio=1, zeroline=True)
style_figure(fig, "Part 1(b): Asset loadings in the PC1-PC2 plane", height=640).show()

### Interpretation of Loadings

The loading vectors form an approximately circular pattern in the PC1-PC2 plane.

The assets appear to be arranged almost uniformly around the circle, resembling the geometry of the 12th roots of unity.

This indicates a highly symmetric covariance structure in which no single asset dominates the factor space.

The first two principal components capture a two-dimensional latent structure that organizes the assets according to their relative phase around the circle.

Rotation Function

In [13]:
def rotate(loadings, angle_deg):

    theta = np.deg2rad(angle_deg)

    R = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    return loadings @ R.T

Rotate by 45 Degrees

In [14]:
rotated_loadings = rotate(
    loadings,
    45
)

Rotated plot

In [15]:
# Before/after rotation shown in one interactive figure.
rotated_df = pd.DataFrame({
    "asset_id": returns_mon.columns,
    "Rotated axis 1": rotated_loadings[:, 0],
    "Rotated axis 2": rotated_loadings[:, 1],
})

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Original PC axes", "Same subspace after 45° rotation"),
    horizontal_spacing=0.10,
)

fig.add_trace(go.Scatter(
    x=loadings_df["PC1 loading"],
    y=loadings_df["PC2 loading"],
    mode="markers+text",
    text=loadings_df["asset_id"],
    textposition="top center",
    marker=dict(size=13, color=loadings_df["angle_deg"], colorscale="Turbo", showscale=False),
    name="Original loadings",
    hovertemplate="%{text}<br>PC1=%{x:.4f}<br>PC2=%{y:.4f}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=rotated_df["Rotated axis 1"],
    y=rotated_df["Rotated axis 2"],
    mode="markers+text",
    text=rotated_df["asset_id"],
    textposition="top center",
    marker=dict(size=13, color=loadings_df["angle_deg"], colorscale="Turbo", showscale=False),
    name="Rotated loadings",
    hovertemplate="%{text}<br>Rot1=%{x:.4f}<br>Rot2=%{y:.4f}<extra></extra>",
), row=1, col=2)

for col in [1, 2]:
    fig.add_hline(y=0, line_width=1, line_dash="dash", opacity=0.6, row=1, col=col)
    fig.add_vline(x=0, line_width=1, line_dash="dash", opacity=0.6, row=1, col=col)
    fig.update_xaxes(scaleanchor=f"y{col}" if col > 1 else "y", scaleratio=1, row=1, col=col)

fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="Rotated axis 1", row=1, col=2)
fig.update_yaxes(title_text="Rotated axis 2", row=1, col=2)
style_figure(fig, "Part 1(b): Rotation changes axis interpretation, not the 2D factor subspace", height=600).show()

### Rotation
Although the rotated factors span exactly the same two-dimensional subspace, the economic interpretation of the individual axes becomes less clear. In the original PCA basis, PC1 and PC2 can be interpreted separately as distinct factors. After rotation, each rotated axis is a linear combination of both PCs, so the original interpretations are mixed across the new coordinates. The underlying factor space is unchanged, but the labeling of individual factors is no longer unique.

After applying a 45° rotation, the coordinates of the assets change, but the overall geometric arrangement remains unchanged.

Pairwise distances, relative positions, and the circular structure are preserved under rotation.

This illustrates the rotational ambiguity of factor models: different coordinate systems can describe the same underlying factor structure.

Therefore, the interpretation should focus on the geometry of the loading configuration rather than the specific orientation of the axes.

In [16]:
# Centre Monday returns

X_centered = (
    returns_mon.values
    - returns_mon.values.mean(axis=0)
)

# Scores on original PCs

scores_original = (
    X_centered @ loadings
)

In [17]:
scores_rotated = (
    X_centered @ rotated_loadings
)

Variance comparison

In [18]:
var_pc1 = np.var(
    scores_original[:,0],
    ddof=1
)

var_pc2 = np.var(
    scores_original[:,1],
    ddof=1
)

var_rot1 = np.var(
    scores_rotated[:,0],
    ddof=1
)

var_rot2 = np.var(
    scores_rotated[:,1],
    ddof=1
)

print("Original Variances")
print(f"PC1 : {var_pc1:.8e}")
print(f"PC2 : {var_pc2:.8e}")

print()

print("Rotated Variances")
print(f"Rot1: {var_rot1:.8e}")
print(f"Rot2: {var_rot2:.8e}")

print()

print(
    "Original Total Variance:",
    var_pc1 + var_pc2
)

print(
    "Rotated Total Variance:",
    var_rot1 + var_rot2
)

print(
    "Lambda1 + Lambda2:",
    eigenvalues[0] + eigenvalues[1]
)

Original Variances
PC1 : 6.09324458e-06
PC2 : 3.00449354e-06

Rotated Variances
Rot1: 4.54886906e-06
Rot2: 4.54886906e-06

Original Total Variance: 9.097738122950913e-06
Rotated Total Variance: 9.097738122950915e-06
Lambda1 + Lambda2: 9.097738122950912e-06



### Rotation variance diagnostic

The next plot makes the rotation-ambiguity point visual: rotating the two-factor subspace redistributes variance across the two displayed axes, but the total variance explained by the two-dimensional subspace remains unchanged.


In [19]:

variance_df = pd.DataFrame({
    "Axis": ["Original PC1", "Original PC2", "Rotated axis 1", "Rotated axis 2"],
    "Variance": [var_pc1, var_pc2, var_rot1, var_rot2],
    "Basis": ["Original PCA axes", "Original PCA axes", "Rotated axes", "Rotated axes"],
})

total_df = pd.DataFrame({
    "Basis": ["Original PCA axes", "Rotated axes", "λ1 + λ2"],
    "Total variance": [var_pc1 + var_pc2, var_rot1 + var_rot2, eigenvalues[0] + eigenvalues[1]],
})

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Per-axis variance changes", "Joint two-factor variance is preserved"),
    horizontal_spacing=0.12,
)

for basis in variance_df["Basis"].unique():
    part = variance_df[variance_df["Basis"] == basis]
    fig.add_trace(go.Bar(
        x=part["Axis"],
        y=part["Variance"],
        name=basis,
        text=[f"{v:.2e}" for v in part["Variance"]],
        textposition="outside",
        hovertemplate="%{x}<br>Variance=%{y:.3e}<extra></extra>",
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=total_df["Basis"],
    y=total_df["Total variance"],
    name="Total variance",
    text=[f"{v:.2e}" for v in total_df["Total variance"]],
    textposition="outside",
    hovertemplate="%{x}<br>Total variance=%{y:.3e}<extra></extra>",
    showlegend=False,
), row=1, col=2)

fig.update_yaxes(title_text="Variance", row=1, col=1)
fig.update_yaxes(title_text="Total variance", row=1, col=2)
style_figure(fig, "Part 1(b): Rotation ambiguity quantified", height=560).show()


#### **Has the joint figure changed?**
No. The total variance explained stays at 9.0977e-06 before and after rotation. Rotation just redistributes variance between the two axes, it does not create or destroy any.
#### **Has your interpretation of PC1 and PC2 changed?**
Yes. Before rotation, PC1 clearly looks like a market factor and PC2 like a dispersion factor. After rotation both axes mix these two signals equally, so neither has a clean interpretation anymore.
#### **What does this tell us about PCA factors versus true factors?**
PCA can tell you how many factors exist and how much variance they jointly explain, but it cannot tell you what those factors actually represent economically. Any rotation of the same subspace is equally valid mathematically, so the labels "PC1" and "PC2" are just one of infinitely many valid descriptions of the same underlying structure.

Compare distances before and after rotation.

In [20]:
original_radius = np.sqrt(
    loadings[:,0]**2 +
    loadings[:,1]**2
)

rotated_radius = np.sqrt(
    rotated_loadings[:,0]**2 +
    rotated_loadings[:,1]**2
)

print(
    "Maximum radius difference:",
    np.max(
        np.abs(
            original_radius -
            rotated_radius
        )
    )
)

Maximum radius difference: 5.551115123125783e-17



### Geometry-preservation diagnostic

The rotated loadings have the same distance from the origin as the original loadings. This reinforces that rotation changes the coordinate description of the factors, not the geometry of the loading cloud.


In [21]:

radius_df = pd.DataFrame({
    "asset_id": returns_mon.columns,
    "Original radius": original_radius,
    "Rotated radius": rotated_radius,
})
radius_long = radius_df.melt(id_vars="asset_id", var_name="Coordinate system", value_name="Radius")

fig = px.bar(
    radius_long,
    x="asset_id",
    y="Radius",
    color="Coordinate system",
    barmode="group",
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Part 1(b): Asset distance from origin is preserved by rotation",
)
fig.update_xaxes(title_text="Asset")
fig.update_yaxes(title_text="Loading radius")
style_figure(fig, height=520).show()


This proves rotation preserves geometry.

Part 1 c

Load Tuesday Data

In [22]:
########################################
# (c) Information leakage analysis
########################################

ticks_tue = load_tick_file("ticks_tue.csv")

returns_tue = (
    ticks_tue
    .pivot(
        index="timestamp",
        columns="asset_id",
        values="mid_return"
    )
    .dropna()
)

print("Tuesday shape:", returns_tue.shape)

Tuesday shape: (28245, 12)


Funtion

In [23]:
def projected_explained_variance(
    test_X,
    eigenvectors_k,
    train_mean
):

    X_centered = test_X - train_mean

    scores = X_centered @ eigenvectors_k.T

    reconstructed = scores @ eigenvectors_k

    residual = X_centered - reconstructed

    ev_ratio = (
        1
        - np.sum(residual**2)
        /
        np.sum(X_centered**2)
    )

    return ev_ratio

Clean procedure

In [24]:
# K chosen from Part 1(a)

K = 3

train_mean = (
    returns_mon.values.mean(axis=0)
)

ev_test_clean = (
    projected_explained_variance(
        returns_tue.values,
        pca_mon.components_[:K],
        train_mean
    )
)

print(
    "Clean EV:",
    ev_test_clean
)

Clean EV: 0.8670429083756577


Leaky Procedure

In [25]:
returns_combined = pd.concat(
    [returns_mon, returns_tue]
)

pca_combined = PCA().fit(
    returns_combined.values
)

combined_mean = (
    returns_combined.values.mean(axis=0)
)

ev_test_leaky = (
    projected_explained_variance(
        returns_tue.values,
        pca_combined.components_[:K],
        combined_mean
    )
)

print(
    "Leaky EV:",
    ev_test_leaky
)

Leaky EV: 0.8670790470563441


Leakage gap

In [26]:
leakage_gap = (
    ev_test_leaky
    - ev_test_clean
)

print(
    "Leakage Gap:",
    leakage_gap
)

Leakage Gap: 3.6138680686415015e-05


In [27]:
summary = pd.DataFrame({
    "Method": [
        "Clean",
        "Leaky"
    ],
    "Explained Variance": [
        ev_test_clean,
        ev_test_leaky
    ]
})

summary

,Method,Explained Variance
0,Clean,0.867043
1,Leaky,0.867079


In [28]:
summary_plot = summary.copy()
summary_plot["Explained Variance %"] = 100 * summary_plot["Explained Variance"]

fig = px.bar(
    summary_plot,
    x="Method",
    y="Explained Variance %",
    color="Method",
    text=summary_plot["Explained Variance %"].map(lambda x: f"{x:.2f}%"),
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Part 1(c): Information leakage inflates Tuesday explained variance",
)
fig.update_traces(textposition="outside")
fig.add_annotation(
    x=0.5,
    y=max(summary_plot["Explained Variance %"]) * 1.02,
    text=f"Leakage gap = {100*leakage_gap:.2f} percentage points",
    showarrow=False,
    font=dict(size=14),
)
fig.update_yaxes(title_text="Projected explained variance on Tuesday (%)", range=[0, max(summary_plot["Explained Variance %"]) * 1.15])
fig.update_xaxes(title_text="Procedure")
style_figure(fig, height=500).show()

### Information Leakage Analysis

1. The projected explained variance obtained using the clean procedure is 0.8670, while the leaky procedure produces 0.8671.

2. The leakage gap is approximately 3.61 × 10⁻⁵, which is positive but extremely small.

**Q. Under what data-generating conditions would the leakage gap be largest? Smallest? How does this relate to the regime change between Monday and Tuesday?**

The leakage gap would be largest when Monday and Tuesday have very different factor structures, such as after a major market event or volatility shift. In that case, including Tuesday data helps PCA adapt to the new regime and artificially improves performance.

The gap would be smallest when Monday and Tuesday have very similar covariance structures. In our results, the gap is very small, suggesting only a minor regime change between the two days.

**Q. Does the leaky procedure over-state or under-state the out-of-sample residual variance? How does this affect the desk's perception of risk?**


The leaky procedure under-states the residual variance because it explains more of Tuesday's variation using information from Tuesday itself. This makes the desk believe that the factor model captures more risk than it actually does, leading to an overly optimistic view of portfolio risk.

**Q3. Why does the fact that PCA has no labels not make it safe from information leakage?**

Even though PCA is unsupervised, its factor directions are still learned from the data. If Tuesday observations are used while estimating those directions, future information has leaked into the model. Therefore, PCA can suffer from information leakage just like supervised methods.


Part 1 d
Proof of Neutrality

In [29]:
w_eq = np.ones(12) / 12

beta_PC1 = float(w_eq @ loadings[:,0])
beta_PC2 = float(w_eq @ loadings[:,1])

print(f"Equal-weight loading on PC1: {beta_PC1:.3e}")
print(f"Equal-weight loading on PC2: {beta_PC2:.3e}")

Equal-weight loading on PC1: 2.820e-04
Equal-weight loading on PC2: 3.234e-04



### Empirical neutrality visualization

The empirical equal-weight exposures are close to zero. The remaining tiny non-zero values are sampling noise around the population polygon result proved in the markdown below.


In [30]:

neutrality_df = pd.DataFrame({
    "Factor": ["PC1", "PC2"],
    "Equal-weight loading": [beta_PC1, beta_PC2],
    "Absolute loading": [abs(beta_PC1), abs(beta_PC2)],
})

fig = px.bar(
    neutrality_df,
    x="Factor",
    y="Equal-weight loading",
    color="Factor",
    text=neutrality_df["Equal-weight loading"].map(lambda x: f"{x:.2e}"),
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Part 1(d): Equal-weight portfolio exposure to discovered PC factors",
)
fig.add_hline(y=0, line_width=1, line_dash="dash")
fig.update_traces(textposition="outside")
fig.update_yaxes(title_text="Portfolio loading")
fig.update_xaxes(title_text="Discovered factor")
style_figure(fig, height=460).show()


The equal-weight portfolio loading on PC1 is 2.820e-04 and on PC2 is 3.234e-04. Both are very close to zero but not exactly zero, which is expected due to sampling noise in a finite dataset.
The equal-weight portfolio assigns weight 1/12 to each asset. Under the assumption that the 12 assets lie at the vertices of a regular 12-gon centred at the origin, the loading of asset k can be represented by the complex number

z_k = e^{i2πk/12},    k = 0,1,...,11

The sum of all 12 vertices is

∑_{k=0}^{11} e^{i2πk/12} = 0

Since the real part of z_k corresponds to the PC1 loading and the imaginary part corresponds to the PC2 loading, taking real and imaginary parts gives

∑_{k=0}^{11} PC1_k = 0

and

∑_{k=0}^{11} PC2_k = 0

The equal-weight portfolio exposure to PC1 is therefore

β_PC1 = (1/12) ∑_{k=0}^{11} PC1_k = 0

Similarly,

β_PC2 = (1/12) ∑_{k=0}^{11} PC2_k = 0

Hence, in the population, the equal-weight portfolio has exactly zero loading on both PC1 and PC2. The small non-zero values observed empirically arise only from sampling noise and finite-sample estimation error.

### Part 1e Observations

1. PCA reveals a strong low-dimensional factor structure in the Monday return data. The first three principal components explain approximately 92.6% of the total variance.

2. All component-selection methods considered (Kaiser criterion, Broken-Stick test, and Parallel Analysis) agree on retaining K = 3 components.

3. The PC1-PC2 loading plot forms an approximately circular pattern, suggesting a highly symmetric covariance structure. Rotating the loading vectors changes the coordinates but preserves the underlying geometry and total variance.

4. The information leakage experiment shows only a very small leakage gap, indicating that Monday and Tuesday have similar factor structures. However, leakage still produces an overly optimistic estimate of model performance and should be avoided.

5. The equal-weight portfolio has near-zero empirical exposure to PC1 and PC2. Under the idealized regular 12-gon geometry, the population exposures are exactly zero because the loading vectors cancel out symmetrically around the origin.


### Final presentation note

All final visuals in this notebook are now Plotly-based and interactive. The key figures to show evaluators are: the scree plot with cumulative variance, the component-selection verdict chart, the PC1-PC2 loading map, the rotation-ambiguity variance chart, the leakage-gap bar chart, and the empirical neutrality exposure chart.
